# 04 - Introduction to MCP

In the previous section, we gave the model access to **Python functions as tools**.

That worked well for simple demos, but those tools lived directly inside the notebook.

## From local tools to external tools

When we want tools to be reusable across applications, notebooks, and assistants, we need a more standard way to expose them.

This is where **MCP** comes in.

MCP stands for **Model Context Protocol**. It gives models a standard way to discover and use external tools.

## What changes?

In the previous notebook, the model received Python functions directly:

```python
tools=[get_weather, search_flights]
```

In this notebook, the model will receive an **MCP session** instead:

```python
tools=[session]
```

The idea stays the same:
- the model gets access to tools
- the model decides whether to use them
- the model combines tool results into a final answer

The difference is that the tools now come from a **server**, not from functions defined in the notebook.

## Setup

Before running the below cells, ensure you have:

1. Authenticated with `gcloud auth application-default login`
2. Set your GCP project and location below
3. Placed `server.py` next to this notebook


In [ ]:
import os
import json
import sys
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path().resolve().parents[1] / ".env")

In [ ]:
# This workshop uses the `google-genai` Python package with Vertex AI.
from google import genai

# Set GCP project and location
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
location = os.getenv("GOOGLE_CLOUD_LOCATION")

# Initialize the genai client for Vertex AI
client = genai.Client(
    vertexai=True, 
    project=project_id, 
    location=location
)

In [ ]:
MODEL_NAME = "gemini-2.5-flash"

SYSTEM_MESSAGE = """
You are a helpful travel assistant.
You have access to tools. Decide whether you need to use a tool.
- If needed, use it
- Otherwise, answer directly
"""

async def ask_llm(
    prompt: str,
    tools: list | None = None,
    system_instruction: str = SYSTEM_MESSAGE,
    temperature: float = 0.7,
):
    config = {
        "system_instruction": system_instruction,
        "temperature": temperature,
    }

    if tools is not None:
        config["tools"] = tools

    response = await client.aio.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )

    return response

## Connect to the MCP server

The MCP server lives in `server.py`.

We start it as a local process and open a client session to it.

That session will act as the model's tool interface.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [ ]:
SERVER_SCRIPT = Path("app/server.py").resolve()

server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_SCRIPT)],
)

mcp_transport = stdio_client(server_params)
read_stream, write_stream = await mcp_transport.__aenter__()

session = ClientSession(read_stream, write_stream)
await session.__aenter__()
await session.initialize()

print("Connected to MCP server.")

## Discover available tools

One benefit of MCP is that the client can ask the server which tools it provides.

This makes the tool interface discoverable and reusable.

In [ ]:
tools = await session.list_tools()

for tool in tools.tools:
    print(f"- {tool.name}: {tool.description}")

## Tool call through MCP

We now ask the same kind of question as before.

The difference is that the model is no longer calling Python functions from the notebook.
It is using tools exposed by the MCP server.

In [ ]:
prompt = """
I am visiting Lisbon this weekend. What should I pack?
"""

response = await ask_llm(
    prompt,
    tools=[session],
)

print(f"\nResponse:\n{response.text}")

## Inspect the tool calls

The final text is useful, but for learning it is even more useful to inspect which tool calls the model decided to make.

In [ ]:
function_calls = [
    {"name": call.name, "args": call.args}
    for call in (response.function_calls or [])
]

print(json.dumps(function_calls, indent=2, ensure_ascii=False))